# Dimensionality reduction

In this exercise, you will use PCA and UMAP to reduce the dimensionality of different kinds of data and compare their properties.

Complete the four functions marked `Your code here` in sections 3–6, then answer the discussion questions in sections 6–9. The plotting code and synthetic datasets are provided. In the student notebook, unfinished functions raise `NotImplementedError`; implement each one before continuing.

Run this notebook in Google Colab.


## 1. Load the data and helpers

In [ ]:
try:
    import google.colab  # type: ignore[import-not-found]
except ImportError:
    pass
else:
    %pip install -q "scipy" "pandas>=2.2" "scikit-learn>=1.4" "matplotlib>=3.8" "pillow>=10" "umap-learn>=0.5.7"


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import umap
from IPython.display import display
from scipy.linalg import orthogonal_procrustes
from scipy.sparse.csgraph import connected_components, shortest_path
from sklearn.decomposition import PCA

plt.style.use("seaborn-v0_8-whitegrid")
RANDOM_STATE = 42

CATEGORY_STYLE = {
    "random": {"color": "#b8b8b8", "marker": "o", "size": 22, "alpha": 0.45},
    "animal": {"color": "#2878b5", "marker": "o", "size": 38, "alpha": 0.90},
    "color": {"color": "#d64f4f", "marker": "s", "size": 38, "alpha": 0.90},
}


def scatter_projection(ax, coordinates, labels, title, words=None, annotate=False):
    """Draw one 2D projection with consistent class colors."""
    for category in ["random", "animal", "color"]:
        mask = labels == category
        if not mask.any():
            continue
        style = CATEGORY_STYLE[category]
        ax.scatter(
            coordinates[mask, 0],
            coordinates[mask, 1],
            c=style["color"],
            marker=style["marker"],
            s=style["size"],
            alpha=style["alpha"],
            label=category,
            edgecolors="white",
            linewidths=0.4,
        )

    if annotate and words is not None:
        for word, (x, y) in zip(words, coordinates):
            ax.annotate(word, (x, y), xytext=(3, 3), textcoords="offset points", fontsize=8)

    ax.set_title(title, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)


Here, we load a CSV containing 10 animal words, 10 color words, and 80 other words, together with their 50-dimensional embeddings from a pretrained **word2vec-like GloVe model** (`glove-wiki-gigaword-50`). The label `random` refers to these hand-picked background words, not randomly generated vectors or a random sample of the vocabulary.

Animal and color words have different meanings, so we will investigate whether their embeddings separate in 2D; this is not guaranteed for every projection.

`X_all` has shape `(100, 50)` and contains one 50-dimensional embedding per word. `words_all` contains the words, and `labels_all` contains `"animal"`, `"color"`, or `"random"`. `X_20`, `words_20`, and `labels_20` contain only the 20 animal and color words.


In [ ]:
EMBEDDING_COLUMNS = [f"embedding_{i}" for i in range(50)]


def load_word_dataset(url):
    """Load and validate the 80/10/10 word dataset and its embeddings."""
    frame = pd.read_csv(url)
    expected_counts = {"random": 80, "animal": 10, "color": 10}
    assert list(frame.columns) == ["word", "category"] + EMBEDDING_COLUMNS
    assert len(frame) == 100
    assert frame["word"].is_unique
    assert frame["category"].value_counts().to_dict() == expected_counts
    return frame


WORD_DATASET_URL = (
    "https://raw.githubusercontent.com/SinitcynLab/INFOMQUDA/"
    "main/assignments/module2/word_dataset.csv"
)
words_df = load_word_dataset(WORD_DATASET_URL)

words_all = words_df["word"].to_numpy()
labels_all = words_df["category"].to_numpy()
X_all = words_df[EMBEDDING_COLUMNS].to_numpy(dtype=np.float32)
assert np.isfinite(X_all).all()

target_mask = labels_all != "random"
words_20 = words_all[target_mask]
labels_20 = labels_all[target_mask]
X_20 = X_all[target_mask]

assert X_all.shape == (100, 50)
assert labels_all.shape == (100,)
assert X_20.shape == (20, 50)

print("X_all:", X_all.shape)
print("labels_all:", labels_all.shape)


## 2. Inspect the data format


In [ ]:
print("One word:", words_all[0])
print("One label:", labels_all[0])
print("One vector shape:", X_all[0].shape)

display(pd.DataFrame({"word": words_all[:5], "label": labels_all[:5]}))
display(pd.DataFrame(X_all[:3, :8], index=words_all[:3]).round(3))


## 3. Random projections

Start with a simple dimensionality reduction method: random projections. Generate a random matrix of shape `(50, 2)` using any method you choose and multiply the embeddings by it to obtain 2D coordinates. Use the `seed` argument so that repeating a seed repeats the projection.

The supplied code calls your function with nine seeds. Compare how clearly each projection separates the animal and color words. You should observe that depending on the seed categories are sometimes separated and sometimes mixed together.




In [ ]:
def random_projection(matrix, seed):
    """Project an (n, d) matrix to 2D using a seeded random matrix."""
    # Your code here.
    raise NotImplementedError("Complete this exercise.")


test_random = random_projection(X_20, seed=RANDOM_STATE)
assert test_random.shape == (20, 2)


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(7, 7))

for ax, seed in zip(axes.flat, range(RANDOM_STATE, RANDOM_STATE + 9)):
    coordinates = random_projection(X_20, seed)
    scatter_projection(ax, coordinates, labels_20, title=f"seed {seed}")

handles, legend_labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, legend_labels, loc="upper center", ncol=2, frameon=False)
fig.suptitle("Nine random projections of the same 20 words", y=0.97)
fig.tight_layout(rect=(0, 0, 1, 0.94));


## 4. PCA implemented with SVD

Now you will implement PCA using SVD. Algorithm reminder:
1. Compute the covariance matrix $C = \frac{1}{n} \sum_i (x_i - \bar{x})(x_i - \bar{x})^T$.
2. Find the $m = 2$ largest eigenvalues $\lambda_1, \lambda_2$ and corresponding eigenvectors $v_1, v_2$ of $C$.
3. Project the data onto the eigenvectors: $y_i = [v_1^T (x_i - \bar{x}), v_2^T (x_i - \bar{x})]$.

You don't need to implement the eigenvalue finder by yourself, instead you can use `np.linalg.svd` to compute the singular value decomposition of the covariance matrix.
The documentation can be found [here](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html). For this real symmetric positive-semidefinite covariance matrix, `U, s, Vt = np.linalg.svd(C)` gives $C = U\,\mathrm{diag}(s)\,V^T$.

**Q1**: What are the eigenvalues and eigenvectors in this decomposition?

Find the two eigenvectors corresponding to the largest eigenvalues and project the data onto them.

Run the cell. Inspect how PCA fitted on the 20 animal and color words separates the two groups.

**Q2** (BONUS): Computing covariance matrix may be bad for the real implementation due to the numerical stability concerns. Instead, SVD can be applied directly to the centered data matrix $X - \bar{X}$, which is more numerically stable. Can you prove why this is equivalent? Hint: show that singular vectors of $X - \bar{X}$ are the eigenvectors of the covariance matrix $C$, then compare how the corresponding singular values relate to the eigenvalues of $C$.



In [ ]:
def pca_svd_projection(matrix, n_components=2):
    """Manual PCA using NumPy's SVD solver."""
    # Your code here.
    raise NotImplementedError("Complete this exercise.")


manual_pca_20 = pca_svd_projection(X_20)
assert manual_pca_20.shape == (20, 2)

fig, ax = plt.subplots(figsize=(6, 4))
scatter_projection(
    ax,
    manual_pca_20,
    labels_20,
    title="Manual PCA via SVD — 20 words",
    words=words_20,
    annotate=True,
)
ax.legend(frameon=False)
fig.tight_layout();


## 5. PCA with scikit-learn

In real data analysis you probably will not implement PCA by yourself.

Implement PCA using [sklearn.decomposition.PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) and compare the results with your implementation. You should see similar results. Principal directions can differ in sign, so a mirrored plot is not an error.


In [ ]:
def pca_projection(matrix, n_components=2):
    """Reduce a matrix with scikit-learn PCA."""
    # Your code here.
    raise NotImplementedError("Complete this exercise.")


sklearn_pca_20 = pca_projection(X_20)
assert sklearn_pca_20.shape == (20, 2)

fig, ax = plt.subplots(figsize=(6, 4))
scatter_projection(
    ax,
    sklearn_pca_20,
    labels_20,
    title="scikit-learn PCA — 20 words",
    words=words_20,
    annotate=True,
)
ax.legend(frameon=False)
fig.tight_layout();


## 6. UMAP with an existing implementation


Use the [UMAP](https://umap-learn.readthedocs.io/en/latest/) implementation from the `umap-learn` package to map the embeddings to 2D. Set `random_state=seed` and `n_jobs=1` for reproducibility, and otherwise start with the default settings.

Compare the results with PCA. What differences do you see in the geometry? When might each method be preferable? Consult the [parameter guide](https://umap-learn.readthedocs.io/en/latest/parameters.html) if you want to experiment further.


In [ ]:
def umap_projection(matrix, seed=RANDOM_STATE):
    """Reduce an (n, d) matrix to two dimensions using seeded UMAP."""
    # Your code here.
    raise NotImplementedError("Complete this exercise.")


umap_20 = umap_projection(X_20)
assert umap_20.shape == (20, 2)

fig, ax = plt.subplots(figsize=(6, 4))
scatter_projection(
    ax,
    umap_20,
    labels_20,
    title="UMAP — 20 words",
    words=words_20,
    annotate=True,
)
ax.legend(frameon=False)
fig.tight_layout();


## 7. Fit on all 100 words versus only 20

Cell below will use your PCA and UMAP functions to fit the embeddings on all 100 words and only on the 20 non-random words.

Compare the results. What do you observe? Does PCA separate the two target categories as clearly when fitted on all 100 words?

**Q3**: Explain any change in terms of the variance that PCA optimizes.



This story is quite important. If you have a set of samples you care about, adding more samples may change the geometry of the data; depending on your setup, it may be advantageous or disadvantageous.


In [ ]:
comparison = [
    ("PCA fit on all 100", pca_projection(X_all), words_all, labels_all),
    ("PCA fit on 20 only", pca_projection(X_20), words_20, labels_20),
    ("UMAP fit on all 100", umap_projection(X_all), words_all, labels_all),
    ("UMAP fit on 20 only", umap_projection(X_20), words_20, labels_20),
]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for ax, (title, coordinates, words, labels) in zip(axes.flat, comparison):
    scatter_projection(
        ax,
        coordinates,
        labels,
        title=title,
        words=words,
        annotate=len(words) == 20,
    )

handles, legend_labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, legend_labels, loc="upper center", ncol=3, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.95));


## 8. PCA versus UMAP

Run the following datasets with PCA and UMAP to see that there is no silver bullet for dimensionality reduction.

The first dataset is a Swiss roll: a 2D sheet rolled into 3D. Compare how PCA and UMAP represent different parts of the roll.

**Q4**: Can you explain why a linear projection can overlap layers, and why a neighborhood-based method can help?



The second dataset is a picture of the letters "PCA" embedded in 100 dimensions using an orthonormal random basis, with Gaussian noise added. Compare how well each method recovers the original picture.

**Q5**: Why is PCA a good match to this construction? Does UMAP preserve the letter shapes, holes, and relative positions?


In [ ]:
from matplotlib import font_manager
from PIL import Image, ImageDraw, ImageFont
from sklearn.datasets import make_swiss_roll


def align_to_reference(points, reference):
    """Align by a similarity transform, preserving shape up to uniform scale."""
    points_centered = points - points.mean(axis=0, keepdims=True)
    reference_centered = reference - reference.mean(axis=0, keepdims=True)
    rotation, _ = orthogonal_procrustes(points_centered, reference_centered)
    rotated = points_centered @ rotation
    scale = np.sum(rotated * reference_centered) / np.sum(rotated**2)
    return rotated * scale + reference.mean(axis=0, keepdims=True)


# Example 1: a nonlinear 2D sheet rolled up in 3D.
swiss_X, swiss_position = make_swiss_roll(
    n_samples=1600, noise=0.05, random_state=RANDOM_STATE
)
swiss_pca = pca_projection(swiss_X)
swiss_umap = umap.UMAP(
    n_components=2,
    n_neighbors=20,
    min_dist=0.10,
    metric="euclidean",
    random_state=RANDOM_STATE,
    n_jobs=1,
).fit_transform(swiss_X)

fig = plt.figure(figsize=(15, 4.5))
axis_3d = fig.add_subplot(1, 3, 1, projection="3d")
axis_pca = fig.add_subplot(1, 3, 2)
axis_umap = fig.add_subplot(1, 3, 3)
axis_3d.scatter(
    swiss_X[:, 0], swiss_X[:, 1], swiss_X[:, 2],
    c=swiss_position, cmap="turbo", s=7, alpha=0.75,
)
axis_3d.view_init(elev=18, azim=-72)
axis_3d.set_title("Swiss roll in 3D")
axis_pca.scatter(*swiss_pca.T, c=swiss_position, cmap="turbo", s=7)
axis_pca.set_title("PCA: layers overlap")
axis_umap.scatter(*swiss_umap.T, c=swiss_position, cmap="turbo", s=7)
axis_umap.set_title("UMAP: neighborhood-based layout")
for axis in [axis_3d, axis_pca, axis_umap]:
    axis.set_xticks([])
    axis.set_yticks([])
    axis.grid(False)
axis_3d.set_zticks([])
fig.tight_layout()
plt.show()


# Example 2: sample a 2D picture, embed it linearly in 100D, and add noise.
picture_rng = np.random.default_rng(RANDOM_STATE)
canvas = Image.new("L", (430, 150), color=255)
font_path = font_manager.findfont(
    font_manager.FontProperties(family="DejaVu Sans", weight="bold")
)
font = ImageFont.truetype(font_path, size=115)
draw = ImageDraw.Draw(canvas)
text_box = draw.textbbox((0, 0), "PCA", font=font)
text_position = (
    (canvas.width - (text_box[2] - text_box[0])) // 2,
    (canvas.height - (text_box[3] - text_box[1])) // 2 - text_box[1],
)
draw.text(text_position, "PCA", font=font, fill=0)
row, column = np.where(np.asarray(canvas) < 128)
chosen = picture_rng.choice(len(row), size=2200, replace=False)
picture_2d = np.column_stack([column[chosen], -row[chosen]]).astype(float)
picture_2d -= picture_2d.mean(axis=0, keepdims=True)
picture_2d /= np.ptp(picture_2d, axis=0).max()

random_basis, _ = np.linalg.qr(picture_rng.normal(size=(100, 2)))
picture_100d = picture_2d @ random_basis.T
picture_100d += 0.015 * picture_rng.normal(size=picture_100d.shape)
picture_pca = align_to_reference(pca_projection(picture_100d), picture_2d)
picture_umap_raw = umap.UMAP(
    n_components=2,
    n_neighbors=10,
    min_dist=0.03,
    metric="euclidean",
    random_state=RANDOM_STATE,
    n_jobs=1,
).fit_transform(picture_100d)
picture_umap = align_to_reference(
    picture_umap_raw,
    picture_2d,
)
point_color = picture_2d[:, 0]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, coordinates, title in [
    (axes[0], picture_2d, "Original 2D picture"),
    (axes[1], picture_pca, "PCA from noisy 100D"),
    (axes[2], picture_umap, "UMAP: local structure, warped layout"),
]:
    axis.scatter(*coordinates.T, c=point_color, cmap="viridis", s=5)
    axis.set_title(title)
    axis.set_aspect("equal", adjustable="datalim")
    axis.set_xticks([])
    axis.set_yticks([])
    axis.grid(False)
fig.tight_layout()
plt.show()


## 9. Bonus: Europe from flight durations

UMAP and other methods can use a precomputed distance matrix instead of assuming Euclidean distance between points. To demonstrate it, we will try to recover the map of Europe based on the flight durations between cities as a distance metric.

We use selected European city pairs from [EDJNet's route dataset](https://github.com/EDJNet/european_routes), estimate the duration of a direct flight on each pair, and compute shortest travel times through this undirected network. We assume that each leg costs 45 minutes of fixed overhead plus cruise time at 800 km/h.

The code starts with UMAP's default neighborhood and spacing settings (`n_neighbors=15`, `min_dist=0.1`), with a fixed seed and precomputed distances. Try changing parameters to improve the result. You may also compare another method, such as MDS, Isomap, or t-SNE. State what you mean by "better" reconstruction. Which failure modes do you observe, and why?

This is an open-ended research-like task. Explore different methods and parameter settings, and analyze the results critically. Try to understand how different assumptions of different methods lead to different reconstructions.

In [ ]:
FLIGHT_DATA_URL = (
    "https://raw.githubusercontent.com/EDJNet/european_routes/"
    "ef44d05cf5f0cf59f635c1f546bd471359f49161/data/train_routes_coords.csv"
)
FLIGHT_OVERHEAD_MINUTES = 45.0
CRUISE_SPEED_KMH = 800.0

LANDMARK_CITIES = {
    "Lisboa", "Porto", "Madrid", "Barcelona", "Malaga",
    "London", "Edinburgh", "Paris", "Bruxelles", "Amsterdam",
    "Berlin", "Frankfurt", "Zürich", "Milano", "Roma",
    "Wien", "Praha", "Warszawa", "Budapest", "Zagreb",
    "Sofia", "București", "København", "Oslo", "Stockholm",
    "Helsinki",
}


def flight_duration_data(frame):
    """Return city names, coordinates, and estimated all-pairs flight times."""
    frame = frame.copy()
    frame[["Origin", "Destination"]] = frame[["Origin", "Destination"]].replace(
        {"Duesseldorf": "Düsseldorf"}
    )
    frame["distance_air_km"] = pd.to_numeric(
        frame["distance_air_km"], errors="coerce"
    )
    frame = frame.dropna(
        subset=[
            "Origin", "Destination", "distance_air_km",
            "origin_latitude", "origin_longitude",
            "destination_latitude", "destination_longitude",
        ]
    )
    frame["flight_minutes"] = (
        FLIGHT_OVERHEAD_MINUTES
        + 60 * frame["distance_air_km"] / CRUISE_SPEED_KMH
    )

    all_cities = sorted(set(frame["Origin"].dropna()) | set(frame["Destination"].dropna()))
    index = {city: position for position, city in enumerate(all_cities)}
    direct = np.full((len(all_cities), len(all_cities)), np.inf)
    np.fill_diagonal(direct, 0.0)
    coordinates = {}

    for row in frame.itertuples():
        i, j = index[row.Origin], index[row.Destination]
        direct[i, j] = direct[j, i] = min(direct[i, j], row.flight_minutes)
        coordinates[row.Origin] = (row.origin_longitude, row.origin_latitude)
        coordinates[row.Destination] = (
            row.destination_longitude, row.destination_latitude
        )

    edge_graph = np.isfinite(direct).astype(float)
    np.fill_diagonal(edge_graph, 0.0)
    _, component_ids = connected_components(edge_graph, directed=False)
    largest_id = np.argmax(np.bincount(component_ids))
    keep = np.flatnonzero(component_ids == largest_id)

    cities = np.array(all_cities)[keep]
    durations = shortest_path(direct[np.ix_(keep, keep)], directed=False)
    map_coordinates = np.array([coordinates[city] for city in cities], dtype=float)
    assert np.isfinite(durations).all()
    return cities, map_coordinates, durations


def align_for_display(points, reference):
    """Remove arbitrary rotation, reflection, scale, and translation."""
    points_centered = points - points.mean(axis=0, keepdims=True)
    reference_centered = reference - reference.mean(axis=0, keepdims=True)
    rotation, _ = orthogonal_procrustes(points_centered, reference_centered)
    rotated = points_centered @ rotation
    scale = np.sum(rotated * reference_centered) / np.sum(rotated**2)
    return rotated * scale + reference.mean(axis=0, keepdims=True)


flight_routes = pd.read_csv(FLIGHT_DATA_URL)
cities, longitude_latitude, flight_durations = flight_duration_data(flight_routes)
# Approximate geographic reference in km, used only for display/evaluation.
longitude, latitude = np.deg2rad(longitude_latitude).T
actual_map = 6371.0 * np.column_stack([
    (longitude - longitude.mean()) * np.cos(latitude.mean()),
    latitude - latitude.mean(),
])
np.testing.assert_allclose(flight_durations, flight_durations.T)
np.testing.assert_allclose(np.diag(flight_durations), 0)
assert (flight_durations >= 0).all()

flight_umap = umap.UMAP(
    n_components=2,
    metric="precomputed",
    n_neighbors=15,
    min_dist=0.1,
    random_state=RANDOM_STATE,
    n_jobs=1,
).fit_transform(flight_durations)
flight_umap_aligned = align_for_display(flight_umap, actual_map)
print(f"Using {len(cities)} connected cities from {len(flight_routes)} source rows.")

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

for ax, coordinates, title in [
    (axes[0], flight_umap_aligned, "Flight-time UMAP, aligned for display"),
    (axes[1], actual_map, "Geographic reference (approx. km)"),
]:
    ax.scatter(coordinates[:, 0], coordinates[:, 1], s=16, color="#5b78c7", alpha=0.8)
    for city, (x, y) in zip(cities, coordinates):
        if city in LANDMARK_CITIES:
            ax.annotate(city, (x, y), xytext=(2, 2), textcoords="offset points", fontsize=7)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal", adjustable="datalim")
    ax.grid(False)

fig.suptitle("Alignment only changes rotation, reflection, scale, and position.")
fig.tight_layout(rect=(0, 0, 1, 0.94));


Route and coordinate data: [EDJNet/european_routes](https://github.com/EDJNet/european_routes). Flight durations are simple estimates from air distance, not scheduled times.
